# Step 1- Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

print("Imports OK ")

Imports OK 


# Step 2 - Configuration

In [8]:
#  Paths 
data_path  = Path("../data/features/feature_matrix.csv")
output_dir  = Path("../data/split")
output_dir.mkdir(parents=True, exist_ok=True)

#  Constants 
nintervals   = 96          # quarter-hours per day for QH
initial_train = 100         # days in first training window
eval_days     = 44          # out-of-sample evaluation days


features = [
    "price_lag1d",
    "price_lag7d",
    "price_hourly_lag1d",
    "price_hourly_lag7d",
    "wind_mwh",
    "solar_mwh",
    "load_mwh",
]
target = "price_eur_mwh"


print(f"Train window : {intial_train} days")
print(f"Eval window  : {eval_days} days")

Train window : 100 days
Eval window  : 44 days


# Step 3 - Load Feature Matrix 

In [9]:
#load feature matrix
df = pd.read_csv(data_path, parse_dates=["timestamp"])
#to ensure the data is sorted by time
df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Shape        : {df.shape}")
print(f"Date range   : {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")
print(f"Unique days  : {df['timestamp'].dt.date.nunique()}")
print()
print(df[features + [target]].describe().round(2))

Shape        : (13824, 9)
Date range   : 2025-10-08 -> 2026-02-28
Unique days  : 144

       price_lag1d  price_lag7d  price_hourly_lag1d  price_hourly_lag7d  \
count     13824.00     13824.00            13824.00            13824.00   
mean         99.05        98.37              111.50              109.73   
std          40.79        42.52               64.86               65.18   
min          -1.05        -5.09              -15.69              -15.69   
25%          82.37        82.20               83.04               81.02   
50%          95.60        95.47              108.39              106.22   
75%         113.06       113.18              132.21              130.09   
max         508.38       508.38              936.28              936.28   

       wind_mwh  solar_mwh  load_mwh  price_eur_mwh  
count  13824.00   13824.00  13824.00       13824.00  
mean    5243.16     792.25  14466.75          98.47  
std     2773.42    1470.75   2196.09          40.65  
min      111.14       

# Step 4 — DST(Daylight Saving Time) Correction

The DST correction for 26 October 2025 was applied in the Feature Engineering notebook before saving. The feature matrix therefore already contains exactly 13,824 rows, corresponding to 144 complete days, each with 96 intervals. The check below confirms this

In [10]:
# vrify all days have exactly 96 intervals
counts   = df.groupby(df["timestamp"].dt.date).size()
bad_days = counts[counts != nintervals]

if len(bad_days) == 0:
    print(f"All {len(counts)} days have exactly {nintervals} intervals")
else:
    print(f"Unexpected days found")
    print(bad_days.to_string())

All 144 days have exactly 96 intervals


# Step 5 - Train / Test Split
The train/test split follows the expanding-window evaluation scheme described in Section 3.4. The feature matrix is divided into an initial training window of 100 days and an out-of-sample evaluation window of 44 days

In [12]:
days       = sorted(df["timestamp"].dt.date.unique())
n_days     = len(days)

train_days = days[:initial_train]
# test_days = the out-of-sample evaluation window
# called "evaluation window" in output to reflect the 
# rolling forecast origin scheme used in models.ipynb
test_days  = days[initial_train:]

print("  TRAIN – TEST SPLIT SUMMARY")
print()
print(f"Total days:  {n_days}")
print(f"Training window: {len(train_days)} days")
print(f"From: {train_days[0]}")
print(f"To: {train_days[-1]}")
print(f"Observations: {len(train_days) * nintervals:,}")
print()
print(f"Evaluation window: {len(test_days)} days")
print(f"From: {test_days[0]}")
print(f"To: {test_days[-1]}")
print(f"Observations: {len(test_days) * nintervals:,}")
print()
print(f"Total observations: {n_days * nintervals:,}")

  TRAIN – TEST SPLIT SUMMARY

Total days:  144
Training window: 100 days
From: 2025-10-08
To: 2026-01-15
Observations: 9,600

Evaluation window: 44 days
From: 2026-01-16
To: 2026-02-28
Observations: 4,224

Total observations: 13,824


# Step 6 — Save the Split

In [14]:
# Save corrected feature matrix
df.to_csv(output_dir / "feature_matrix_clean.csv", index=False)
print(f"Saved: feature_matrix_clean.csv  ({len(df):,} rows)")

# Save train day list 
pd.Series([str(d) for d in train_days]).to_csv(output_dir / "train_days.csv", index=False, header=["date"])
print(f"Saved: train_days.csv({len(train_days)} days)")

# Save test day list 
pd.Series([str(d) for d in test_days]).to_csv(output_dir / "test_days.csv", index=False, header=["date"])
print(f"Saved: test_days.csv({len(test_days)} days)")

Saved: feature_matrix_clean.csv  (13,824 rows)
Saved: train_days.csv(100 days)
Saved: test_days.csv(44 days)


# Step 7 — Final Verification
checks two things: no data leakage and no missing values

In [15]:
# No data leakage check
last_train = train_days[-1]
first_test = test_days[0]
gap_days = (pd.Timestamp(first_test) - pd.Timestamp(last_train)).days
print(f"Last training day: {last_train}")
print(f"First test day: {first_test}")
print(f"Gap between them: {gap_days} day(s)")
if gap_days == 1:
    print("no data leakage")
else:
    print("check split")

Last training day: 2026-01-15
First test day: 2026-01-16
Gap between them: 1 day(s)
no data leakage
